# Radial grid refinement via stages

In this notebook we showcase how to perform radial grid refinement in GVEC by utilizing the `stages` functionality.

`stages` allows one to chain several GVEC runs together, always *restarting* from the previous solution. This option is especially useful, when each new *restart* only changes a few parameters from an original parameter set.

```{seealso}
See the [stages section in the GVEC user guide](/user/stages)([&#x1F310;](https://gvec.readthedocs.io/latest/user/stages.html)) for more details.
```

We start again with the necessary imports and set the number of threads to use.

In [ ]:
# set the number of OMP threads for gvec (needs to be before import of gvec)
import os

os.environ["OMP_NUM_THREADS"] = "2"

In [ ]:
import gvec

Next we simply load the parameters from the `.toml` file written in the [elliptic tokamak tutorial](./010_tokamak)([&#x1F310;](https://gvec.readthedocs.io/latest/tutorials/notebooks/010_tokamak.html)), and adapt them instead of specifying everything again.

In [ ]:
params = gvec.util.read_parameters("elliptok_parameters.toml")

## Add  stages to parameter dictionary

Note that:
- The `stages` are specified as a list of dictionaries.
- Each stage restarts from the solution of the previous stage.
- If a parameter is specified within a stage, it overwrites the parameter from the main dictionary.
- After the stage is passed, all parameters are reset to ones defined in the main dictionary

In [ ]:
params["stages"] = [
    {"sgrid_nElems": 2, "minimize_tol": 5e-7},
    {"sgrid_nElems": 15, "minimize_tol": 2e-7},
    {"sgrid_nElems": 40, "minimize_tol": 1e-7},
]

The three stages defined above will ramp up `minimize_tol` and also increase the number of radial elements from $2$ to $15$ to $40$. Therefore, each stage tries to find a better converged and more refined equilibrium solution. This ramping can help improve runtime, especially when the initial guess for the equilibrium is poor, e.g. during a *cold* start.

For potential use later on we can also write `params` again into a parameter file.

In [ ]:
gvec.util.write_parameters(
    params, "radial_refinement_parameters.toml"
)

Lets run GVEC with the new stages! Note the change in the screen output compared to the previous runs.

In [ ]:
runpath = "run_radial_refinement"
run = gvec.run(params, runpath=runpath)

- To check everything went as expected we again have a look at the diagnostics.
- In the visualization, the stages are indicated and the quantities are now plotted over the accumulated GVEC iteration number.

In [ ]:
fig = run.plot_diagnostics_minimization()

The spikes in the forces after a restart can be attributed to the interpolation to the refined grid.

---

## Run without stages

Let us now compare this to a run without refinement stages. For this we just copy `params` and set the relevant parameters to the ones of the final stage.

In [ ]:
params2 = params.copy()
del params2["stages"]
params2["sgrid_nElems"] = 40
params2["minimize_tol"] = 1e-7

runpath = "no_radial_refinement"
run = gvec.run(params2, runpath=runpath)

fig2 = run.plot_diagnostics_minimization()

As we can see, even for this simple tokamak case we save iterations and runtime using the ramping via `stages` since the main cost is in the iterations with the highest resolution.

To summarize, the `stages` functionality allows for running similar GVEC cases in succession and change only some few parameters. This can be used, for example, to perform radial refinement which might improve runtime.